# Imports set up rendering

In [1]:
#@title Check if MuJoCo installation was successful

import distutils.util
import os
import subprocess
if subprocess.run('nvidia-smi').returncode:
  raise RuntimeError(
      'Cannot communicate with GPU. '
      'Make sure you are using a GPU Colab runtime. '
      'Go to the Runtime menu and select Choose runtime type.')

# Add an ICD config so that glvnd can pick up the Nvidia EGL driver.
# This is usually installed as part of an Nvidia driver package, but the Colab
# kernel doesn't install its driver via APT, and as a result the ICD is missing.
# (https://github.com/NVIDIA/libglvnd/blob/master/src/EGL/icd_enumeration.md)
NVIDIA_ICD_CONFIG_PATH = '/usr/share/glvnd/egl_vendor.d/10_nvidia.json'
if not os.path.exists(NVIDIA_ICD_CONFIG_PATH):
  with open(NVIDIA_ICD_CONFIG_PATH, 'w') as f:
    f.write("""{
    "file_format_version" : "1.0.0",
    "ICD" : {
        "library_path" : "libEGL_nvidia.so.0"
    }
}
""")

# Configure MuJoCo to use the EGL rendering backend (requires GPU)
print('Setting environment variable to use GPU rendering:')
# %env MUJOCO_GL=egl
os.environ['MUJOCO_GL'] = 'egl'
os.environ['PYOPENGL_PLATFORM'] = 'egl'
try:
  print('Checking that the installation succeeded:')
  import mujoco
  mujoco.MjModel.from_xml_string('<mujoco/>')
except Exception as e:
  raise e from RuntimeError(
      'Something went wrong during installation. Check the shell output above '
      'for more information.\n'
      'If using a hosted Colab runtime, make sure you enable GPU acceleration '
      'by going to the Runtime menu and selecting "Choose runtime type".')

print('Installation successful.')

# Tell XLA to use Triton GEMM, this improves steps/sec by ~30% on some GPUs
os.environ["XLA_FLAGS"] = (
    "--xla_gpu_enable_triton_softmax_fusion=true "
    "--xla_gpu_triton_gemm_any=True "
    # "--xla_gpu_enable_async_collectives=true "
    # "--xla_gpu_enable_latency_hiding_scheduler=true "
    # "--xla_gpu_enable_highest_priority_async_stream=true "
)
os.environ['CUDA_VISIBLE_DEVICES'] = '1' # Use GPU 1

Tue Jan 14 15:46:54 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.107.02             Driver Version: 550.107.02     CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX A6000               On  |   00000000:41:00.0 Off |                  Off |
| 37%   65C    P2            117W /  300W |    1440MiB /  49140MiB |     99%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
%load_ext autoreload
%autoreload 2
import os

# os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"] = "0.95"
# os.environ["CUDA_VISIBLE_DEVICES"] = "1"  # Use GPU 1
import functools
import jax
# jax.config.update("jax_enable_x64", True)

n_gpus = jax.device_count(backend="gpu")
print(f"Using {n_gpus} GPUs")
from typing import Dict
from brax import envs
import mujoco
import pickle
import warnings
import mediapy as media
import hydra
import jax.numpy as jp
import numpy as np 
import matplotlib.pyplot as plt

from omegaconf import DictConfig, OmegaConf
# from brax.training.agents.ppo import networks as ppo_networks
# from custom_brax import custom_ppo as ppo
# from custom_brax import custom_wrappers
# from custom_brax import custom_ppo_networks
# from orbax import checkpoint as ocp
# from flax.training import orbax_utils
# from preprocessing.mjx_preprocess import process_clip_to_train
# from envs.Fly_Env_Brax import FlyTracking, FlyMultiClipTracking
# from utils.utils import *
# from utils.fly_logging import log_eval_rollout
from tqdm import tqdm
import brax.math as brax_math
warnings.filterwarnings("ignore", category=DeprecationWarning)
# jax.config.update("jax_enable_x64", True)

from hydra import initialize, compose
from hydra.core.hydra_config import HydraConfig
from hydra.core.global_hydra import GlobalHydra
from pathlib import Path

Using 1 GPUs


# Cleaning Data

In [3]:
%matplotlib widget

In [ ]:
path_to_config = './configs'
with initialize(version_base=None, config_path=path_to_config):
    cfg=compose(config_name='config.yaml',overrides=[],return_hydra_config=True,)
    HydraConfig.instance().set_config(cfg)
    


In [10]:
mjcf_path = 'PATH_TO_MJCF'
spec = mujoco.MjSpec()
spec = spec.from_file(mjcf_path)
mj_model = spec.compile()

mj_data = mujoco.MjData(mj_model)

site_names = [
'tracking[coxa_T1_right]',
'tracking[femur_T1_right]',
'tracking[tibia_T1_right]',
'tracking[tarsus_T1_right]',
'tracking[claw_T1_right]',
'tracking[coxa_T2_right]',
'tracking[femur_T2_right]',
'tracking[tibia_T2_right]',
'tracking[tarsus_T2_right]',
'tracking[claw_T2_right]',
'tracking[coxa_T3_right]',
'tracking[femur_T3_right]',
'tracking[tibia_T3_right]',
'tracking[tarsus_T3_right]',
'tracking[claw_T3_right]',
]

site_idxs = [
    mujoco.mj_name2id(mj_model, mujoco.mju_str2Type("site"), site)
    for site in site_names
]
mujoco.mj_forward(mj_model, mj_data)
ref_bout = mj_data.site_xpos[site_idxs]
ref_bout = ref_bout - ref_bout[0]

    
def procrustes_jax(source_points, target_points, floor=None):
    """
    Perform Procrustes analysis to align matrix source_points to matrix target_points.
    
    Parameters:
    source_points : (N, M) array_like
        The source input matrix.
    target_points : (N, M) array_like
        The reference/target input matrix.
    kp_idxs: (N,) array_like
        The indices of the keypoints to use for the alignment.
    floor : float, optional
    
    Returns:
    d : float
        The residual sum of squared errors.
    Z : (T,N, M) ndarray
        The matrix Y, transformed to best match X.
    R : (M, M) ndarray
        The rotation applied to Y.
    scale : float
        The scaling factor applied to Y.
    """
    
    # Center both matrices
    source_centered = source_points - jp.mean(source_points, axis=0)
    target_centered = target_points - jp.mean(target_points, axis=0)

    # Normalize the Frobenius norm to 1
    target_norm = jp.linalg.norm(target_centered)
    source_norm = jp.linalg.norm(source_centered)
    target_centered /= target_norm
    source_centered /= source_norm

    # Compute the optimal rotation matrix using Singular Value Decomposition (SVD) for the first frame
    U, _, Vt = jp.linalg.svd(jp.dot(target_centered.T, source_centered).T)
    R = jp.dot(U, Vt)

    # Apply the rotation and scaling to all frames
    scale = target_norm / source_norm
    Z = jp.dot(source_points, R) * scale
    d = jp.sum(jp.square(target_centered - Z))

    return d, Z, R, scale

def align_groundplane(Z0, percentile=10, floor_height=-0.13):
    """
    Align the keypoints to the groundplane plane by rotating and translating the keypoints.

    Args:
        Z0 (_type_): _description_

    Returns:
        _type_: _description_
    """
    # Compute the optimal rotation matrix for all frames of endeff to groundplane
    Z1 = Z0[:,:,2].copy()
    bot_10 = jp.percentile(Z1, percentile, axis=0)
    bot_t = Z0[Z1 < bot_10]
    target_points = bot_t.copy()
    groundplane = target_points.at[:,2].set(floor_height)

    source_points = bot_t.copy()
    target_points = groundplane.copy()
    source_centered = source_points - jp.mean(source_points, axis=0)
    target_centered = target_points - jp.mean(target_points, axis=0)

    # Compute the optimal rotation matrix for all frames of endeff to groundplan
    H = jp.dot(source_centered.T, target_centered)
    U, _, Vt = jp.linalg.svd(H.T)
    R2 = jp.dot(U, Vt)

    # Ensure a proper rotation (det(R) = 1, avoiding reflection)
    if jp.linalg.det(R2) < 0:
        Vt = jp.array(Vt)
        Vt = Vt.at[-1, :].set(-1*Vt[-1, :])
        R2 = jp.dot(U, Vt)
    t = jp.mean(target_points, axis=0) - jp.dot(R2, jp.mean(source_points, axis=0))

    # Apply the rotation and translation to all frames, all kp
    Z2 = jp.dot(Z0, R2) + t
    return Z2, R2, t, bot_t, groundplane


def plot_plane_and_points(points, fig=None, ax=None, color='r', label=''):
    """
    Plot the plane and the 3D points.
    
    Args:
        points (jax.numpy.ndarray): Array of shape (N, 3), the 3D points.
    """
    print('points',points.shape)
    
    # Compute the centroid of the points
    centroid = jp.mean(points, axis=0)
    
    # Center the points around the centroid
    centered_points = points - centroid
    
    # Compute the SVD of the centered points
    _, _, vh = jp.linalg.svd(centered_points, full_matrices=False)
    
    # The plane's normal is the singular vector corresponding to the smallest singular value
    normal = vh[-1]
    
    # Compute the intercept term (d) for the plane equation ax + by + cz + d = 0
    d = -jp.dot(normal, centroid)
    
    # Create a mesh grid for the plane
    x = jp.linspace(points[:, 0].min(), points[:, 0].max(), 10)
    y = jp.linspace(points[:, 1].min(), points[:, 1].max(), 10)
    x, y = jp.meshgrid(x, y)
    
    # Compute the corresponding z values for the plane equation ax + by + cz + d = 0
    z = (-normal[0] * x - normal[1] * y - d) / normal[2]
    
    # Plot the points and the plane
    if fig is None and ax is None:
        fig = plt.figure(figsize=(10, 7))
        ax = fig.add_subplot(111, projection='3d')
    
    # Plot the points
    ax.scatter(points[:, 0], points[:, 1], points[:, 2], c=color, label=label+' Points')
    
    # Plot the plane
    ax.plot_surface(x, y, z, alpha=0.5, color=color, label=label + ' Plane')
    # ax.plot_surface(x, y, -.13*jp.ones_like(x), alpha=0.5, color='black', label='Ground Plane')
    
    # Set labels and title
    ax.set_xlabel("X-axis")
    ax.set_ylabel("Y-axis")
    ax.set_zlabel("Z-axis")
    ax.set_title("Fitted Plane and 3D Points")
    ax.legend()
    

# procrustes_partial = functools.partial(procrustes_jax)
jit_procrustes = jax.jit(procrustes_jax)
vmap_procrustes_partial = jax.vmap(jit_procrustes, (None,0))


In [9]:
import pandas as pd

##### Change stuff to load your data #####
base_path = Path('/data/users/eabe/biomech_model/Flybody/datasets/Tuthill_data/')
data_path = base_path / 'combined_wt_berlin_walking_v3.pq'
full_df = pd.read_parquet(data_path, engine='pyarrow')

legs = ['T1_left', 'T1_right', 'T2_left', 'T2_right', 'T3_left', 'T3_right']
joints = ['coxa', 'femur', 'tibia', 'tarsus']
xpos_geoms = ['coxa', 'femur', 'tibia', 'tarsus', 'claw']
joint_names = [f'{joint}_{leg}' for leg in legs for joint in joints]
xpos_names = [f'{joint}_{leg}' for leg in legs for joint in xpos_geoms]
# physics.named.data.framepos[pos_names]
site_names = [f'tracking[{joint_name}]' for joint_name in xpos_names]

legs_data = ['L1', 'R1', 'L2','R2', 'L3','R3']
joints_data = ['A','B','C','D','E']
coords_data = ['_x','_y','_z']
joint_pos_columns = [leg + joint + coord 
                     for leg in legs_data
                     for joint in joints_data 
                     for coord in coords_data]

def transform_bout(bout):
    """Transform a single frame from data to model reference frame."""
    # Rotate around z-axis.
    bout = bout[:, :, [1, 0, 2]]
    bout[:,:, 1] *= -1
    # bout = bout - bout[:,:1,:]
    # Change units mm to cm.
    bout *= 0.1
    return bout

# coxa_kp_idxs = np.arange(0,30,5)
# endeff_kp_idxs = np.arange(4,30,5)
# straight_bouts = bout_stats[fast & straight].index
# straight_bout_num = np.array([i[0] for i in straight_bouts],dtype=int)
# mean_straight_bout = (full_df[full_df['walking_bout_number'] == 14574][joint_pos_columns].values.reshape(-1,30, 3)).mean(axis=0)
# # ref_bout = transform_bout(full_df[full_df['walking_bout_number'] == 14574][joint_pos_columns].values.reshape(-1,30, 3))
all_bout_nums = full_df['walking_bout_number'].unique()
# # bout_dict = {'walking_bout{:04}'.format(n):{} for n in range(len(all_bout_nums))}
for n, bout_num in enumerate(tqdm(all_bout_nums)):
    bout = full_df[full_df['walking_bout_number'] == bout_num]
#     kp_data = transform_bout(bout[joint_pos_columns].values.reshape(-1,30, 3))
#     # kp_data = np.array([procrustes(ref_bout, kp_data[i])[1] for i in range(kp_data.shape[0])])
#     # kp_data = np.array(vmap_procrustes_partial(ref_bout,kp_data))
#     kp_data = kp_data - kp_data[:,:1,:]
#     d, _, R, scale  = procrustes_jax(jp.mean(kp_data,axis=0),ref_bout[:])
#     Z = jp.dot(kp_data, R) * scale
#     Z0 = Z[:,endeff_kp_idxs,:]
#     _, R2, t, bot_t, Ground = align_groundplane(Z0,percentile=10)
#     Z2 = jp.dot(Z, R2.T) + t
#     bout_dict['walking_bout{:04}'.format(n)]['orig_xpos'] = np.array(Z2).copy()
    # bout_dict['walking_bout{:04}'.format(n)]['orig_xpos'] = np.array(kp_data).copy()
# ioh5.save('/data/users/eabe/biomech_model/Flybody/datasets/Tuthill_data/all_walking_bouts.h5', bout_dict)
# bout_dict = ioh5.load('/data/users/eabe/biomech_model/Flybody/datasets/Tuthill_data/all_walking_bouts.h5')


100%|██████████| 1730/1730 [00:02<00:00, 797.30it/s]


In [12]:
coxa_kp_idxs = np.arange(0,30,5)
endeff_kp_idxs = np.arange(4,30,5)
n = 1729
kp_data = # Enter your kp_data here
d, _, R, scale  = procrustes_jax(jp.mean(kp_data,axis=0),ref_bout[:])
Z = jp.dot(kp_data, R.T) * scale
Z0 = Z[:,endeff_kp_idxs,:]
_, R2, t, bot_t, Ground = align_groundplane(Z0,percentile=10)
Z2 = jp.dot(Z, R2.T) + t

In [13]:
plt.close('all')

In [ ]:
import matplotlib as mpl
def map_discrete_cbar(cmap,N):
    cmap = plt.get_cmap(cmap,N+1)
    bounds = np.arange(-.5,N+1)
    norm = mpl.colors.BoundaryNorm(bounds, cmap.N)
    return cmap, norm
cmap,norm = map_discrete_cbar('turbo',6)
fly_skel = np.array([(n,n+1) for n in range(29)])
fly_skel = np.stack((fly_skel[:4],fly_skel[5:9],fly_skel[10:14],fly_skel[15:19],fly_skel[20:24],fly_skel[25:30],))

source_points = kp_data[0,coxa_kp_idxs]
target_points = ref_bout[coxa_kp_idxs]

fig = plt.figure(figsize=(10, 7))
ax = fig.add_subplot(111, projection='3d')
plot_plane_and_points(source_points.reshape(-1,3), fig, ax, color='b', label='Source')
plot_plane_and_points(target_points.reshape(-1,3), fig, ax, color='k', label='target')
plot_plane_and_points(Z[:,coxa_kp_idxs].reshape(-1,3), fig, ax, color='r', label='transform')

# ax.scatter(ref_bout[::5, 0], ref_bout[::5, 1],ref_bout[::5, 2], c=np.arange(ref_bout[::5].shape[0]),cmap=cmap)
# ax.scatter(Z[::5, 0], Z[::5, 1],Z[::5, 2], c=np.arange(Z[::5].shape[0]),cmap=cmap)
# ax.scatter(Z[:,4::5, 0], Z[:,4::5, 1],Z[:,4::5, 2],c='r')
for n in range(fly_skel.shape[0]):
    ax.plot(ref_bout[fly_skel[n], 0], ref_bout[fly_skel[n], 1],  ref_bout[fly_skel[n], 2],c='k')
    ax.plot(np.mean(Z[:,fly_skel[n], 0],axis=0), np.mean(Z[:,fly_skel[n], 1],axis=0),np.mean(Z[:,fly_skel[n], 2],axis=0),c='r')

In [ ]:
Z2b = jp.dot(bot_t, R2.T) + t
fig = plt.figure(figsize=(10, 7))
ax = fig.add_subplot(111, projection='3d')
# plot_plane_and_points(bot_t.reshape(-1,3), fig, ax, color='b', label='Z')
plot_plane_and_points(Ground.reshape(-1,3), fig, ax, color='k', label='ground')
plot_plane_and_points(Z2b[:,:].reshape(-1,3), fig, ax, color='r', label='Z2')
plot_plane_and_points(Z2[:,endeff_kp_idxs].reshape(-1,3), fig, ax, color='r', label='Z2')

for n in range(fly_skel.shape[0]):
    ax.plot(ref_bout[fly_skel[n], 0], ref_bout[fly_skel[n], 1],  ref_bout[fly_skel[n], 2],c='k')
    ax.plot(np.mean(Z2[:,fly_skel[n], 0],axis=0), np.mean(Z2[:,fly_skel[n], 1],axis=0),np.mean(Z[:,fly_skel[n], 2],axis=0),c='r')

In [ ]:
# bout_dict2 = ioh5.load('/data/users/eabe/biomech_model/Flybody/datasets/Tuthill_data/all_walking_bouts.h5')
# ioh5.save('/data/users/eabe/biomech_model/Flybody/datasets/Tuthill_data/all_walking_bouts.h5', bout_dict)